# FM1 실습 3 확장: PDF·Excel 파일을 활용한 RAG 파이프라인

## 학습 목표
- 실제 파일(PDF, Excel)에서 지식 베이스를 구성하는 방법을 배운다
- 다양한 파일 형식을 하나의 RAG 파이프라인에 통합하는 방법을 익힌다
- 한국어 문서 처리를 포함한 실용적인 RAG 시스템을 구현한다

## 확장된 RAG 흐름
```
[파일 로드]  PDF/Excel → 텍스트 추출 → 청크 분할 → 임베딩 → 벡터 저장
                                                              ↓
[질문 시]    질문 → 임베딩 → 유사 청크 검색 → Claude에 전달 → 답변
```

## 사용하는 파일
| 파일 | 형식 | 내용 |
|------|------|------|
| `techkorea_소개서.pdf` | PDF (한국어) | 회사 소개, 비전, 사업 분야 |
| `techkorea_HR데이터.xlsx` | Excel (한국어) | 복지포인트, 경조사, 건강검진 |
| 내부 텍스트 | Python 문자열 | 기존 HR 정책 6개 항목 |

## 0단계: 환경 준비

In [ ]:
# 한글 폰트 설치 (PDF 생성에 필요)
!apt-get install -y fonts-nanum -q 2>/dev/null

# 필요한 라이브러리 설치
!pip install anthropic sentence-transformers numpy pdfplumber reportlab openpyxl pandas scikit-learn -q

print("✅ 설치 완료!")

In [ ]:
import getpass
import numpy as np
import pandas as pd
import pdfplumber
import openpyxl
import anthropic
import matplotlib
import matplotlib.pyplot as plt
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 한글 폰트 설정 (그래프용)
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 임포트 완료!")

In [ ]:
api_key = getpass.getpass("🔑 Anthropic API 키를 입력하세요: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ Claude API 준비 완료!")

## 1단계: 샘플 파일 생성

실습용 한국어 PDF와 Excel 파일을 생성합니다.  
> 💡 실제 업무에서는 이 단계를 건너뛰고 본인 파일을 바로 업로드하면 됩니다.

In [ ]:
# ── 한국어 PDF 생성 ──────────────────────────────────────────
# reportlab에 NanumGothic 폰트 등록
pdfmetrics.registerFont(TTFont('Nanum',     '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'))
pdfmetrics.registerFont(TTFont('NanumBold', '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'))

title_style   = ParagraphStyle('Title',   fontName='NanumBold', fontSize=18, leading=26, spaceAfter=16)
heading_style = ParagraphStyle('Heading', fontName='NanumBold', fontSize=13, leading=20, spaceAfter=6)
body_style    = ParagraphStyle('Body',    fontName='Nanum',     fontSize=11, leading=17, spaceAfter=4)

# PDF 내용: TechKorea 회사 소개서
sections = [
    ("회사 개요",
     "TechKorea는 2010년 설립된 AI 및 클라우드 솔루션 전문 기업입니다. "
     "서울 강남구에 본사를 두고 있으며, 현재 500명의 직원이 근무하고 있습니다. "
     "코스닥 상장 기업으로 연매출 1,000억원 규모를 달성했습니다."),
    ("비전과 미션",
     "비전: '기술로 세상을 연결한다'. "
     "미션: 혁신적인 AI 기술로 기업과 사회의 디지털 전환을 이끕니다. "
     "2030년까지 아시아 Top 10 AI 기업 진입을 목표로 합니다."),
    ("주요 사업 분야",
     "첫째, AI 솔루션: 자연어처리·컴퓨터비전·예측분석 서비스를 제공합니다. "
     "둘째, 클라우드 서비스: 기업용 SaaS·PaaS 플랫폼을 운영합니다. "
     "셋째, 데이터 컨설팅: 빅데이터 분석 및 인사이트 도출 서비스를 제공합니다."),
    ("조직 문화",
     "TechKorea는 수평적 조직 문화를 지향합니다. "
     "직급 대신 영어 이름으로 호칭하며 자유로운 의견 개진을 장려합니다. "
     "매주 금요일 오후는 개인 연구 및 사이드 프로젝트 시간으로 운영됩니다. "
     "분기별 해커톤을 개최하여 우수 아이디어를 실제 제품화하는 기회를 제공합니다."),
    ("채용 정보",
     "TechKorea는 AI 엔지니어, 데이터 사이언티스트, 클라우드 아키텍트, "
     "프로덕트 매니저를 상시 채용합니다. "
     "채용 절차: 서류 → 코딩테스트 → 기술면접 → 최종면접. "
     "신입사원은 3개월 온보딩 프로그램을 통해 업무 적응을 지원받습니다."),
]

PDF_PATH = "techkorea_소개서.pdf"
doc = SimpleDocTemplate(PDF_PATH, pagesize=A4,
                        rightMargin=60, leftMargin=60, topMargin=60, bottomMargin=60)
story = [Paragraph("TechKorea 회사 소개서", title_style)]
for heading, body in sections:
    story.append(Paragraph(heading, heading_style))
    story.append(Paragraph(body, body_style))
    story.append(Spacer(1, 8))
doc.build(story)

print(f"✅ PDF 생성 완료: {PDF_PATH}")
print(f"   {len(sections)}개 섹션 포함")

In [ ]:
# ── 한국어 Excel 생성 (다중 시트) ───────────────────────────
from openpyxl import Workbook

wb = Workbook()

# 시트 1: 복지포인트 제도
ws1 = wb.active
ws1.title = "복지포인트"
ws1.append(["항목", "연간포인트(원)", "사용방법", "비고"])
ws1.append(["건강관리",  300000, "병원, 약국, 헬스장",       "가족 포함 사용 가능"])
ws1.append(["자기계발",  200000, "책, 강의, 자격증",          "영수증 제출 필요"])
ws1.append(["여가활동",  150000, "영화, 공연, 스포츠 관람",   "본인만 사용 가능"])
ws1.append(["식비지원",  100000, "식당, 카페, 배달앱",        "월 한도 있음"])
ws1.append(["교통비",    120000, "대중교통, 택시",             "출퇴근 및 업무용 포함"])
ws1.append(["총 합계",   870000, "연간 총 복지포인트",         "매년 1월 초기화"])

# 시트 2: 경조사 지원
ws2 = wb.create_sheet("경조사지원")
ws2.append(["경조사 유형", "지원금액(원)", "추가 지원", "신청 기간"])
ws2.append(["본인 결혼",         500000, "결혼휴가 5일",     "결혼일 기준 30일 이내"])
ws2.append(["자녀 출생",         300000, "출산휴가/육아휴직 별도", "출생일 기준 30일 이내"])
ws2.append(["본인 부모 사망",    300000, "경조휴가 5일",     "사망일 기준 7일 이내"])
ws2.append(["배우자 부모 사망",  200000, "경조휴가 3일",     "사망일 기준 7일 이내"])
ws2.append(["자녀 결혼",         200000, "경조휴가 1일",     "결혼일 기준 30일 이내"])

# 시트 3: 건강검진 프로그램
ws3 = wb.create_sheet("건강검진")
ws3.append(["검진 종류", "대상", "지원금액(원)", "주기", "예약 방법"])
ws3.append(["기본검진",   "전 직원",          300000, "연 1회", "제휴 병원 직접 예약"])
ws3.append(["종합검진",   "5년 이상 재직자",  500000, "연 1회", "HR 시스템 신청"])
ws3.append(["배우자검진", "기혼 직원 배우자", 200000, "연 1회", "제휴 병원 직접 예약"])
ws3.append(["암검진",     "40세 이상",        200000, "연 1회", "HR 시스템 신청"])
ws3.append(["치과검진",   "전 직원",          100000, "연 1회", "제휴 치과 직접 예약"])

EXCEL_PATH = "techkorea_HR데이터.xlsx"
wb.save(EXCEL_PATH)

print(f"✅ Excel 생성 완료: {EXCEL_PATH}")
print(f"   시트 1 '복지포인트':  {ws1.max_row - 1}개 항목")
print(f"   시트 2 '경조사지원':  {ws2.max_row - 1}개 항목")
print(f"   시트 3 '건강검진':    {ws3.max_row - 1}개 항목")

## 2단계: 파일에서 지식 베이스 구성

각 파일 형식에 맞는 **파서(parser)**를 만들어 텍스트를 추출합니다.  
추출된 텍스트는 `청크(chunk)` 단위로 나누어 저장합니다.

> **청크란?** RAG에서 검색 단위가 되는 텍스트 조각입니다. 너무 짧으면 맥락이 부족하고, 너무 길면 검색 정확도가 떨어집니다.

In [ ]:
def load_pdf_as_chunks(pdf_path):
    """
    PDF 파일을 열어 페이지별로 텍스트를 추출하고,
    줄 단위로 묶어 의미 있는 청크를 반환합니다.
    """
    chunks = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text:
                continue

            lines = [l.strip() for l in text.split('\n') if l.strip()]

            # 짧은 줄(제목 추정)을 만날 때마다 새 청크로 분리
            current_lines = []
            for line in lines:
                is_heading = len(line) < 20  # 20자 미만이면 제목으로 간주
                if is_heading and current_lines:
                    content = " ".join(current_lines)
                    if len(content) > 40:
                        chunks.append({
                            "id":      f"pdf_p{page_num+1}_{len(chunks)}",
                            "title":   f"[PDF] {current_lines[0][:25]}",
                            "content": content,
                            "source":  "PDF",
                            "file":    pdf_path,
                        })
                    current_lines = [line]  # 새 청크 시작
                else:
                    current_lines.append(line)

            # 마지막 청크 저장
            if current_lines:
                content = " ".join(current_lines)
                if len(content) > 40:
                    chunks.append({
                        "id":      f"pdf_p{page_num+1}_{len(chunks)}",
                        "title":   f"[PDF] {current_lines[0][:25]}",
                        "content": content,
                        "source":  "PDF",
                        "file":    pdf_path,
                    })
    return chunks


pdf_chunks = load_pdf_as_chunks(PDF_PATH)

print(f"✅ PDF 로드 완료: {len(pdf_chunks)}개 청크")
for c in pdf_chunks:
    print(f"  - {c['title']}: {c['content'][:50]}...")

In [ ]:
def load_excel_as_chunks(excel_path):
    """
    Excel 파일의 모든 시트를 읽어 각 시트를 하나의 청크로 변환합니다.
    헤더 + 각 행을 자연어 문장처럼 이어 붙여 임베딩에 적합한 형태로 만듭니다.
    """
    chunks = []
    xl = pd.ExcelFile(excel_path)

    for sheet_name in xl.sheet_names:
        df = xl.parse(sheet_name)
        columns = df.columns.tolist()

        # 각 행을 "컬럼: 값, 컬럼: 값" 형태 문자열로 변환
        row_texts = []
        for _, row in df.iterrows():
            row_text = ", ".join(
                f"{col}: {val}" for col, val in zip(columns, row) if pd.notna(val)
            )
            row_texts.append(row_text)

        # 시트 전체를 하나의 청크로 (행이 적을 때)
        if len(row_texts) <= 8:
            chunks.append({
                "id":      f"excel_{sheet_name}",
                "title":   f"[Excel] {sheet_name}",
                "content": f"[{sheet_name}] " + " | ".join(row_texts),
                "source":  "Excel",
                "file":    excel_path,
            })
        else:
            # 행이 많으면 5행씩 나누어 저장
            for i in range(0, len(row_texts), 5):
                group = row_texts[i:i+5]
                chunks.append({
                    "id":      f"excel_{sheet_name}_{i}",
                    "title":   f"[Excel] {sheet_name} ({i+1}~{min(i+5, len(row_texts))}행)",
                    "content": " | ".join(group),
                    "source":  "Excel",
                    "file":    excel_path,
                })
    return chunks


excel_chunks = load_excel_as_chunks(EXCEL_PATH)

print(f"✅ Excel 로드 완료: {len(excel_chunks)}개 청크")
for c in excel_chunks:
    print(f"  - {c['title']}: {c['content'][:60]}...")

In [ ]:
# ── 기존 텍스트 지식 베이스 (FM1_03 내용, source 필드 추가) ──
text_knowledge_base = [
    {
        "id": "vacation", "title": "[정책] 연차 휴가", "source": "텍스트", "file": "내부정책",
        "content": "TechKorea 직원은 입사 1년 후 연 15일의 유급 연차를 부여받습니다. "
                   "연차는 매년 1월 1일 자동 지급되며, 당해 연도 미사용 연차의 50%는 다음 연도로 이월할 수 있습니다. "
                   "연차 신청은 최소 3일 전에 HR 시스템을 통해 신청하며, 팀장의 승인 후 확정됩니다."
    },
    {
        "id": "remote", "title": "[정책] 재택근무", "source": "텍스트", "file": "내부정책",
        "content": "TechKorea는 주 3일 출근, 2일 재택근무를 기본 정책으로 합니다. "
                   "재택근무일은 팀별 협의로 지정하며, 화요일과 목요일을 공통 출근일로 권장합니다. "
                   "신입사원의 경우 입사 후 3개월간은 전일 출근을 원칙으로 합니다. "
                   "재택근무 시 오전 9시~오후 6시 코어타임을 준수해야 합니다."
    },
    {
        "id": "benefits", "title": "[정책] 복리후생", "source": "텍스트", "file": "내부정책",
        "content": "TechKorea의 복리후생 프로그램: 중식 및 석식 무료 제공(구내식당), "
                   "건강검진 연 1회 지원(배우자 포함), 자녀 학자금 지원(중·고등학교), "
                   "사내 피트니스 센터 무료 이용, 명절 상품권 지급(설·추석), "
                   "장기근속 포상(5·10·15년 마일스톤 보상)."
    },
    {
        "id": "equipment", "title": "[정책] 장비 지원", "source": "텍스트", "file": "내부정책",
        "content": "신규 입사자에게 노트북(맥북 프로 또는 ThinkPad X1 선택), 모니터 27인치, "
                   "무선 키보드/마우스를 기본 지급합니다. 장비 교체 주기는 3년이며, "
                   "개인 업무 필요에 따라 추가 장비(패드, 외장 SSD 등)는 팀장 승인 후 신청 가능합니다."
    },
    {
        "id": "expense", "title": "[정책] 경비 처리", "source": "텍스트", "file": "내부정책",
        "content": "업무 관련 경비는 지출 후 30일 이내에 경비 시스템(ExPro)에 영수증 첨부하여 신청합니다. "
                   "식비: 1인 2만원 한도(고객 접대 시 5만원), 교통비: 실비 처리, "
                   "숙박비: 서울/수도권 15만원, 지방 10만원 한도. 5만원 초과 경비는 팀장 사전 승인이 필요합니다."
    },
    {
        "id": "training", "title": "[정책] 교육 지원", "source": "텍스트", "file": "내부정책",
        "content": "TechKorea는 직원 역량 개발을 위해 연간 200만원의 교육비를 지원합니다. "
                   "지원 대상: 직무 관련 외부 교육, 자격증, 도서, 온라인 강의. "
                   "AI/데이터 관련 교육은 별도 예산으로 연 100만원 추가 지원합니다."
    },
]

# ── 모든 소스 통합 ────────────────────────────────────────────
knowledge_base = text_knowledge_base + pdf_chunks + excel_chunks

print("📚 통합 지식 베이스 구성 완료!")
print(f"  텍스트 정책:  {len(text_knowledge_base)}개")
print(f"  PDF 청크:     {len(pdf_chunks)}개")
print(f"  Excel 청크:   {len(excel_chunks)}개")
print(f"  총합:         {len(knowledge_base)}개 청크")

# 출처별 분포 시각화
source_counts = pd.Series([c['source'] for c in knowledge_base]).value_counts()
colors = ['#4CAF50', '#2196F3', '#FF9800']

plt.figure(figsize=(7, 3.5))
bars = plt.bar(source_counts.index, source_counts.values, color=colors[:len(source_counts)])
for bar, val in zip(bars, source_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(val), ha='center', va='bottom', fontweight='bold')
plt.title('지식 베이스 출처별 청크 수')
plt.xlabel('출처')
plt.ylabel('청크 수')
plt.tight_layout()
plt.show()

## 3단계: 임베딩 생성

모든 청크를 숫자 벡터(임베딩)로 변환하여 저장합니다.  
한국어를 지원하는 다국어 모델 `paraphrase-multilingual-MiniLM-L12-v2`를 사용합니다.

In [ ]:
print("임베딩 모델 로딩 중...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

texts = [item['content'] for item in knowledge_base]
kb_embeddings = embed_model.encode(texts, show_progress_bar=True)

print(f"\n✅ 임베딩 완료!")
print(f"  벡터 행렬 크기: {kb_embeddings.shape}")
print(f"  → {kb_embeddings.shape[0]}개 청크 × {kb_embeddings.shape[1]}차원 벡터")

## 4단계: 검색 및 답변 생성

검색 결과에 **출처 정보**를 포함시켜 어떤 파일에서 답변했는지 추적할 수 있게 합니다.

In [ ]:
def retrieve(query, top_k=3):
    """질문과 가장 관련 있는 청크를 출처 정보와 함께 반환합니다."""
    query_vec = embed_model.encode([query])
    sims = cosine_similarity(query_vec, kb_embeddings)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    return [
        {
            "title":      knowledge_base[i]['title'],
            "content":    knowledge_base[i]['content'],
            "source":     knowledge_base[i]['source'],
            "file":       knowledge_base[i]['file'],
            "similarity": float(sims[i]),
        }
        for i in top_indices
    ]


def answer_with_rag(query, verbose=True):
    """검색된 문서를 컨텍스트로 Claude에 전달하여 답변을 생성합니다."""
    retrieved = retrieve(query)

    if verbose:
        print("🔍 검색된 관련 문서:")
        for r in retrieved:
            print(f"  [{r['source']}] {r['title']}  (유사도: {r['similarity']:.3f})")
        print()

    context = "\n\n".join(
        f"[출처: {r['source']} | {r['title']}]\n{r['content']}"
        for r in retrieved
    )

    prompt = f"""다음은 TechKorea 회사의 공식 문서(정책서, 회사소개서, HR 데이터)에서 가져온 내용입니다:

{context}

위 문서만을 근거로 아래 질문에 정확하게 답변해주세요.
문서에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요.

질문: {query}"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


print("✅ 검색 및 답변 함수 준비 완료")

## 5단계: 다양한 소스에서 답변 비교

PDF, Excel, 텍스트 각각의 소스에서 답변이 나오는 질문을 테스트합니다.

In [ ]:
# 출처가 다른 질문들로 RAG 테스트
demo_questions = [
    ("TechKorea는 어떤 회사이고 비전이 무엇인가요?",    "→ PDF 소개서에서 답변 예상"),
    ("복지포인트는 연간 얼마이고 어디에 쓸 수 있나요?", "→ Excel 복지포인트 시트에서 답변 예상"),
    ("결혼할 때 경조사 지원금과 휴가는 어떻게 되나요?", "→ Excel 경조사지원 시트에서 답변 예상"),
    ("연차 휴가는 몇 일이고 이월이 가능한가요?",          "→ 텍스트 정책에서 답변 예상"),
]

for question, hint in demo_questions:
    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print(f"   {hint}")
    print(f"{'='*65}")
    print(answer_with_rag(question, verbose=True))

## ✏️ 실습: 나만의 파일 업로드

직접 PDF나 Excel 파일을 업로드하여 지식 베이스에 추가하고 질문해보세요.

**지원 파일 형식:**
- `.pdf` — 텍스트가 내장된 PDF (스캔 이미지 PDF는 OCR 필요)
- `.xlsx`, `.xls` — Excel 파일 (여러 시트 자동 처리)

In [ ]:
from google.colab import files

print("📁 업로드할 파일을 선택하세요 (PDF 또는 Excel)")
uploaded = files.upload()  # 파일 선택 대화상자

for filename in uploaded.keys():
    print(f"\n파일 감지: {filename}")

    if filename.lower().endswith('.pdf'):
        new_chunks = load_pdf_as_chunks(filename)
        print(f"  PDF에서 {len(new_chunks)}개 청크 추출")

    elif filename.lower().endswith(('.xlsx', '.xls')):
        new_chunks = load_excel_as_chunks(filename)
        print(f"  Excel에서 {len(new_chunks)}개 청크 추출")

    else:
        print("  ⚠️ 지원하지 않는 형식입니다. PDF 또는 Excel을 업로드하세요.")
        continue

    # 지식 베이스 & 임베딩 업데이트
    knowledge_base.extend(new_chunks)
    new_vecs = embed_model.encode([c['content'] for c in new_chunks])
    kb_embeddings = np.vstack([kb_embeddings, new_vecs])

    print(f"  ✅ 추가 완료! 현재 지식 베이스: {len(knowledge_base)}개 청크")

# 업로드 후 질문
if uploaded:
    my_question = "업로드한 문서의 내용에 대해 궁금한 것을 여기 입력하세요"
    print(f"\n❓ {my_question}")
    print(answer_with_rag(my_question))

## 📝 정리

### 파일 형식별 처리 방법

| 파일 형식 | 파서 라이브러리 | 청크 전략 | 한국어 지원 |
|-----------|----------------|-----------|------------|
| 텍스트 | 직접 작성 | 주제별 단락 | ✅ |
| PDF | `pdfplumber` | 줄 단위 → 섹션 분리 | ✅ (폰트 내장 필요) |
| Excel | `pandas` + `openpyxl` | 시트별, 행 그룹 | ✅ |

### 핵심 포인트
- **출처 추적**: 각 청크에 `source`, `file` 필드를 저장하면 답변 신뢰도가 높아집니다
- **동적 확장**: 새 파일을 업로드하면 임베딩만 추가하면 되어 재학습이 불필요합니다
- **청크 크기**: 너무 작으면 맥락 손실, 너무 크면 검색 정확도 저하 — 실험적으로 조정하세요

### 실제 적용 시 추가 고려사항
- **스캔 PDF**: 이미지 PDF는 `pytesseract` + `tesseract-ocr-kor`로 OCR 처리 필요
- **복잡한 Excel**: 병합 셀·수식이 많은 파일은 `openpyxl` 직접 파싱 권장
- **대용량 파일**: 청크를 FAISS나 ChromaDB 같은 벡터 DB에 저장하면 속도 향상